In [1]:
import geopandas as gpd
import typer
import os
import glob
import gerrychain
import networkx as nx
import matplotlib.pyplot as plt
from itertools import product
import tqdm
import functools
import sys

### Substantive change #1:
Angle 1 in `pipeline/calculate_metrics.py`. Below is an outer function that calls for _angle_1(). We show the former implementation of _angle_1() and what we changed below.

In [2]:

def angle_1(graph: gerrychain.Graph, x_col: str, y_col: str, lam: float = 1) -> float:
    """
    This implements `<x_col, y_col>` from the paper
    """
    first_summation, second_summation = _angle_1(graph, x_col, y_col)

    if lam == None:
        return first_summation
    else:
        return (lam * first_summation) + second_summation
    

In [3]:
#former implementation
def _angle_1(graph: gerrychain.Graph, x_col: str, y_col: str) -> float:
    first_summation = 0
    second_summation = 0
    for node in graph.nodes():
        first_summation += int(graph.nodes[node][x_col]) * int(graph.nodes[node][y_col])

        for neighbor in graph.neighbors(node): #this double counts the second summation. 
            #The second sum in the paper is over all edges whereas here (i,j) and (j,i) 
            #are counted separately.
            second_summation += int(graph.nodes[node][x_col]) * int(
                graph.nodes[neighbor][y_col]
            )
            second_summation += int(graph.nodes[neighbor][x_col]) * int(
                graph.nodes[node][y_col]
            )

    return (first_summation, second_summation)


In [ ]:
#current implementation
def _angle_1_new(graph: gerrychain.Graph, x_col: str, y_col: str) -> float:
    first_summation = 0
    second_summation = 0
    for node in graph.nodes():
        first_summation += int(graph.nodes[node][x_col]) * int(graph.nodes[node][y_col])

    for node, neighbor in graph.edges():
        second_summation += int(graph.nodes[node][x_col]) * int(
            graph.nodes[neighbor][y_col])
        second_summation += int(graph.nodes[neighbor][x_col]) * int(
            graph.nodes[node][y_col])

### Substantive change #2:
Angle 2 in `pipeline/calculate_metrics.py`. Here also we show the former implementation and the new version of _angle_2()

In [ ]:
def angle_2(graph: gerrychain.Graph, x_col: str, y_col: str, lam: float = 1) -> float:
    """
    This implements `<<x_col, y_col>>` from the paper
    """
    first_summation, second_summation = _angle_2(graph, x_col, y_col)

    if lam == None:
        return first_summation
    else:
        return 0.5 * ((lam * first_summation) + second_summation)

In [ ]:
# old implementation
def _angle_2(graph: gerrychain.Graph, x_col: str, y_col: str, lam: float = 1) -> float:
    first_summation = 0
    second_summation = 0
    for node in graph.nodes():
        first_summation += int(graph.nodes[node][x_col]) * int(
            graph.nodes[node][y_col]
        ) - ((int(graph.nodes[node][x_col]) + int(graph.nodes[node][y_col])) * 0.5)

        for neighbor in graph.neighbors(node): #the same overcounting issue as above
            second_summation += int(graph.nodes[node][x_col]) * int(
                graph.nodes[neighbor][y_col]
            )
            second_summation += int(graph.nodes[neighbor][x_col]) * int(
                graph.nodes[node][y_col]
            )

    return (first_summation, second_summation)

In [ ]:
#new implementation
def _angle_2_new(graph: gerrychain.Graph, x_col: str, y_col: str, lam: float = 1) -> float:
    first_summation = 0
    second_summation = 0
    for node in graph.nodes():
        first_summation += int(graph.nodes[node][x_col]) * int(
            graph.nodes[node][y_col]
        ) - ((int(graph.nodes[node][x_col]) + int(graph.nodes[node][y_col])) * 0.5)

    for node, neighbor in graph.edges():
        second_summation += int(graph.nodes[node][x_col]) * int(
            graph.nodes[neighbor][y_col])
        second_summation += int(graph.nodes[neighbor][x_col]) * int(
            graph.nodes[node][y_col])

### Substantive change #3:
Moran's I old implementation and new.

In [ ]:
#old implementation
def moran(graph: gerrychain.Graph, x_col: str, tot_col: str) -> float:
    # TODO: double/triple-check the 2 coefficient
    total_shares = [] #calculates moran in terms of population shares, 
    #not counts whereas the rest of the metrics take counts
    for node in graph.nodes():
        total_shares.append(
            int(graph.nodes[node][x_col]) / int(graph.nodes[node][tot_col])
        )
    avg = sum(total_shares) / len(total_shares)

    top_summation = 0
    bottom_summation = 0
    for node in graph.nodes():
        node_share = int(graph.nodes[node][x_col]) / int(graph.nodes[node][tot_col])
        bottom_summation += (node_share - avg) ** 2

        for neighbor in graph.neighbors(node):
            neighbor_share = int(graph.nodes[neighbor][x_col]) / int(
                graph.nodes[neighbor][tot_col]
            )
            top_summation += (node_share - avg) * (neighbor_share - avg)

    return (
        (len(graph.nodes()) / len(graph.edges()))
        * (top_summation / bottom_summation)
        * 0.5) #this code has the same double counting error but the 0.5 corrects for it

In [ ]:
def moran_right(graph: gerrychain.Graph, x_col: str, tot_col: str) -> float:
    # Moran but using counts and w/out double counting
    avg = sum(int(graph.nodes[node][x_col]) for node in graph.nodes()) / len(graph.nodes())

    top_summation = 0
    bottom_summation = 0
    for node in graph.nodes():
        bottom_summation += (int(graph.nodes[node][x_col])  - avg) ** 2

    for u, v in graph.edges():
        top_summation += (int(graph.nodes[u][x_col]) - avg) * (int(graph.nodes[v][x_col]) - avg)

    return (
        (len(graph.nodes()) / len(graph.edges()))
        * (top_summation / bottom_summation)
    )